In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

In [2]:
df = pd.read_csv('advertising.csv')
df

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Ad Topic Line,City,Male,Country,Timestamp,Clicked on Ad
0,68.95,35,61833.90,256.09,Cloned 5thgeneration orchestration,Wrightburgh,0,Tunisia,2016-03-27 00:53:11,0
1,80.23,31,68441.85,193.77,Monitored national standardization,West Jodi,1,Nauru,2016-04-04 01:39:02,0
2,69.47,26,59785.94,236.50,Organic bottom-line service-desk,Davidton,0,San Marino,2016-03-13 20:35:42,0
3,74.15,29,54806.18,245.89,Triple-buffered reciprocal time-frame,West Terrifurt,1,Italy,2016-01-10 02:31:19,0
4,68.37,35,73889.99,225.58,Robust logistical utilization,South Manuel,0,Iceland,2016-06-03 03:36:18,0
...,...,...,...,...,...,...,...,...,...,...
995,72.97,30,71384.57,208.58,Fundamental modular algorithm,Duffystad,1,Lebanon,2016-02-11 21:49:00,1
996,51.30,45,67782.17,134.42,Grass-roots cohesive monitoring,New Darlene,1,Bosnia and Herzegovina,2016-04-22 02:07:01,1
997,51.63,51,42415.72,120.37,Expanded intangible solution,South Jessica,1,Mongolia,2016-02-01 17:24:57,1
998,55.55,19,41920.79,187.95,Proactive bandwidth-monitored policy,West Steven,0,Guatemala,2016-03-24 02:35:54,0


In [3]:
# Check data type
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Daily Time Spent on Site  1000 non-null   float64
 1   Age                       1000 non-null   int64  
 2   Area Income               1000 non-null   float64
 3   Daily Internet Usage      1000 non-null   float64
 4   Ad Topic Line             1000 non-null   object 
 5   City                      1000 non-null   object 
 6   Male                      1000 non-null   int64  
 7   Country                   1000 non-null   object 
 8   Timestamp                 1000 non-null   object 
 9   Clicked on Ad             1000 non-null   int64  
dtypes: float64(3), int64(3), object(4)
memory usage: 78.2+ KB


In [4]:
df.describe().round()

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Male,Clicked on Ad
count,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0
mean,65.0,36.0,55000.0,180.0,0.0,0.0
std,16.0,9.0,13415.0,44.0,0.0,1.0
min,33.0,19.0,13996.0,105.0,0.0,0.0
25%,51.0,29.0,47032.0,139.0,0.0,0.0
50%,68.0,35.0,57012.0,183.0,0.0,0.0
75%,79.0,42.0,65471.0,219.0,1.0,1.0
max,91.0,61.0,79485.0,270.0,1.0,1.0


In [5]:
df.describe(include= 'object')

,Ad Topic Line,City,Country,Timestamp
count,1000,1000,1000,1000
unique,1000,969,237,1000
top,Virtual 5thgeneration emulation,Lisamouth,Czech Republic,2016-06-03 21:43:21
freq,1,3,9,1


In [6]:
df.drop(['Ad Topic Line', 'City', 'Timestamp'], axis= 1, inplace= True)

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.isna().sum()

Daily Time Spent on Site    0
Age                         0
Area Income                 0
Daily Internet Usage        0
Male                        0
Country                     0
Clicked on Ad               0
dtype: int64

# Data Preprocessing

### Split Data into Input Features and Target Variable

In [9]:
x = df.drop('Clicked on Ad', axis= 1)
y = df['Clicked on Ad']

### Split Data into Train & Test

In [10]:
y.value_counts()

Clicked on Ad
0    500
1    500
Name: count, dtype: int64

In [11]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.2, random_state= 0, stratify= y)

### Handle Numerical Features

In [12]:
scaling_cols = x_train.columns.drop(['Country', 'Male'])
scaling_cols

Index(['Daily Time Spent on Site', 'Age', 'Area Income',
       'Daily Internet Usage'],
      dtype='object')

In [13]:
from sklearn.preprocessing import RobustScaler

rc = RobustScaler()

x_train[scaling_cols] = rc.fit_transform(x_train[scaling_cols])
x_test[scaling_cols] = rc.transform(x_test[scaling_cols])

In [14]:
x_train.head()

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Male,Country
600,0.594840,1.083333,-0.569248,-0.580319,1,Kyrgyz Republic
737,0.120824,1.416667,-0.863072,-0.750515,0,Sweden
33,-0.459354,-1.000000,-1.483623,0.374008,0,Senegal
519,-1.224016,0.416667,-0.615119,-0.392377,1,Mongolia
341,0.157944,1.083333,-0.246821,-0.840987,0,Mexico


### Handle Categorical Features

In [15]:
# ! pip install category_encoders

In [16]:
from category_encoders import BinaryEncoder

be = BinaryEncoder()

be_train = be.fit_transform(x_train[['Country']])
be_test = be.transform(x_test[['Country']])

In [17]:
be_train

,Country_0,Country_1,Country_2,Country_3,Country_4,Country_5,Country_6,Country_7
600,0,0,0,0,0,0,0,1
737,0,0,0,0,0,0,1,0
33,0,0,0,0,0,0,1,1
519,0,0,0,0,0,1,0,0
341,0,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...
622,1,1,0,0,1,1,0,1
682,0,0,0,1,1,1,0,1
357,0,0,1,0,0,1,1,0
918,1,1,0,1,0,1,0,0


In [18]:
x_train = pd.concat([x_train, be_train], axis= 1).drop('Country', axis = 1)

x_test = pd.concat([x_test, be_test], axis= 1).drop('Country', axis = 1)

In [19]:
x_train

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Male,Country_0,Country_1,Country_2,Country_3,Country_4,Country_5,Country_6,Country_7
600,0.594840,1.083333,-0.569248,-0.580319,1,0,0,0,0,0,0,0,1
737,0.120824,1.416667,-0.863072,-0.750515,0,0,0,0,0,0,0,1,0
33,-0.459354,-1.000000,-1.483623,0.374008,0,0,0,0,0,0,0,1,1
519,-1.224016,0.416667,-0.615119,-0.392377,1,0,0,0,0,0,1,0,0
341,0.157944,1.083333,-0.246821,-0.840987,0,0,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
622,-0.287491,2.083333,-0.603832,-0.388754,1,1,1,0,0,1,1,0,1
682,-0.905902,0.083333,-0.343783,-0.717901,1,0,0,0,1,1,1,0,1
357,-0.673163,0.333333,-0.627936,-0.887973,0,0,0,1,0,0,1,1,0
918,0.401448,-0.250000,0.609726,0.407873,1,1,1,0,1,0,1,0,0


# Machine Learning

### Logistic Regression

In [20]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(penalty= None)

lr.fit(x_train, y_train)

print('Train Accuracy :',lr.score(x_train, y_train) * 100)
print('Test Accuracy :',lr.score(x_test, y_test) * 100)

Train Accuracy : 97.375
Test Accuracy : 95.0


In [54]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(penalty= 'l2', C= 1)

lr.fit(x_train, y_train)

print('Train Accuracy :',lr.score(x_train, y_train) * 100)
print('Test Accuracy :',lr.score(x_test, y_test) * 100)

Train Accuracy : 97.5
Test Accuracy : 94.5


In [22]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors= 3)
knn.fit(x_train, y_train)
print('Train Accuracy :', knn.score(x_train, y_train) * 100)
print('Test Accuracy :', knn.score(x_test, y_test) * 100)

Train Accuracy : 97.125
Test Accuracy : 91.0
